In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_account (1)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_account
(
    account_id INT,

    currency_id INT,
    currency_name STRING,

    account_type STRING,
    name STRING,
    code_store STRING,
    note STRING,

    active BOOLEAN,
    reconcile BOOLEAN,
    non_trade BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_account AS target
USING
(
    SELECT
        CAST(a.id AS INT) AS account_id,

        CAST(GET_JSON_OBJECT(a.currency_id, '$[0]') AS INT) AS currency_id,
        GET_JSON_OBJECT(a.currency_id, '$[1]')               AS currency_name,

        a.account_type,
        a.name,
        a.code_store,
        a.note,

        CAST(a.active AS BOOLEAN)    AS active,
        CAST(a.reconcile AS BOOLEAN) AS reconcile,
        CAST(a.non_trade AS BOOLEAN) AS non_trade,

        a.create_date AS created_at,
        a.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        a.dwh_source_db

    FROM bronze.account_account a
    WHERE a.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_account'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_id = source.account_id

WHEN MATCHED THEN UPDATE SET
    target.currency_id   = source.currency_id,
    target.currency_name = source.currency_name,
    target.account_type  = source.account_type,
    target.name          = source.name,
    target.code_store    = source.code_store,
    target.note          = source.note,
    target.active        = source.active,
    target.reconcile     = source.reconcile,
    target.non_trade     = source.non_trade,
    target.created_at    = source.created_at,
    target.updated_at    = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;



In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_analytic_account (2)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_analytic_account
(
    analytic_account_id INT,

    plan_id INT,
    plan_name STRING,

    company_id INT,
    company_name STRING,

    partner_id INT,
    partner_name STRING,

    code STRING,
    name STRING,

    active BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_analytic_account AS target
USING
(
    SELECT
        CAST(aa.id AS INT) AS analytic_account_id,

        CAST(GET_JSON_OBJECT(aa.plan_id, '$[0]') AS INT)    AS plan_id,
        GET_JSON_OBJECT(aa.plan_id, '$[1]')                  AS plan_name,

        CAST(GET_JSON_OBJECT(aa.company_id, '$[0]') AS INT) AS company_id,
        GET_JSON_OBJECT(aa.company_id, '$[1]')               AS company_name,

        CAST(GET_JSON_OBJECT(aa.partner_id, '$[0]') AS INT) AS partner_id,
        GET_JSON_OBJECT(aa.partner_id, '$[1]')               AS partner_name,

        aa.code,
        aa.name,

        CAST(aa.active AS BOOLEAN) AS active,

        aa.create_date AS created_at,
        aa.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        aa.dwh_source_db

    FROM bronze.account_analytic_account aa
    WHERE aa.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_analytic_account'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.analytic_account_id = source.analytic_account_id

WHEN MATCHED THEN UPDATE SET
    target.plan_id              = source.plan_id,
    target.plan_name            = source.plan_name,
    target.company_id           = source.company_id,
    target.company_name         = source.company_name,
    target.partner_id           = source.partner_id,
    target.partner_name         = source.partner_name,
    target.code                 = source.code,
    target.name                 = source.name,
    target.active               = source.active,
    target.created_at           = source.created_at,
    target.updated_at           = source.updated_at,
    target.dwh_loaded_at        = current_timestamp(),
    target.dwh_source_db        = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_analytic_line (3)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_analytic_line
(
    analytic_line_id INT,

    account_id INT,
    account_name STRING,

    product_id INT,
    product_name STRING,

    product_uom_id INT,
    product_uom_name STRING,

    partner_id INT,
    partner_name STRING,

    company_id INT,
    company_name STRING,

    currency_id INT,
    currency_name STRING,

    journal_id INT,
    journal_name STRING,

    general_account_id INT,
    general_account_name STRING,

    name STRING,
    category STRING,
    code STRING,
    ref STRING,

    amount DOUBLE,
    unit_amount DOUBLE,

    date TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_analytic_line AS target
USING
(
    SELECT
        CAST(al.id AS INT) AS analytic_line_id,

        CAST(GET_JSON_OBJECT(al.account_id, '$[0]') AS INT)          AS account_id,
        GET_JSON_OBJECT(al.account_id, '$[1]')                        AS account_name,

        CAST(GET_JSON_OBJECT(al.product_id, '$[0]') AS INT)          AS product_id,
        GET_JSON_OBJECT(al.product_id, '$[1]')                        AS product_name,

        CAST(GET_JSON_OBJECT(al.product_uom_id, '$[0]') AS INT)      AS product_uom_id,
        GET_JSON_OBJECT(al.product_uom_id, '$[1]')                    AS product_uom_name,

        CAST(GET_JSON_OBJECT(al.partner_id, '$[0]') AS INT)          AS partner_id,
        GET_JSON_OBJECT(al.partner_id, '$[1]')                        AS partner_name,

        CAST(GET_JSON_OBJECT(al.company_id, '$[0]') AS INT)          AS company_id,
        GET_JSON_OBJECT(al.company_id, '$[1]')                        AS company_name,

        CAST(GET_JSON_OBJECT(al.currency_id, '$[0]') AS INT)         AS currency_id,
        GET_JSON_OBJECT(al.currency_id, '$[1]')                       AS currency_name,

        CAST(GET_JSON_OBJECT(al.journal_id, '$[0]') AS INT)          AS journal_id,
        GET_JSON_OBJECT(al.journal_id, '$[1]')                        AS journal_name,

        CAST(GET_JSON_OBJECT(al.general_account_id, '$[0]') AS INT)  AS general_account_id,
        GET_JSON_OBJECT(al.general_account_id, '$[1]')                AS general_account_name,

        al.name,
        al.category,
        al.code,
        al.ref,

        CAST(al.amount AS DOUBLE)      AS amount,
        CAST(al.unit_amount AS DOUBLE) AS unit_amount,

        al.date,

        al.create_date AS created_at,
        al.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        al.dwh_source_db

    FROM bronze.account_analytic_line al
    WHERE al.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_analytic_line'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.analytic_line_id = source.analytic_line_id

WHEN MATCHED THEN UPDATE SET
    target.account_id           = source.account_id,
    target.account_name         = source.account_name,
    target.product_id           = source.product_id,
    target.product_name         = source.product_name,
    target.product_uom_id       = source.product_uom_id,
    target.product_uom_name     = source.product_uom_name,
    target.partner_id           = source.partner_id,
    target.partner_name         = source.partner_name,
    target.company_id           = source.company_id,
    target.company_name         = source.company_name,
    target.currency_id          = source.currency_id,
    target.currency_name        = source.currency_name,
    target.journal_id           = source.journal_id,
    target.journal_name         = source.journal_name,
    target.general_account_id   = source.general_account_id,
    target.general_account_name = source.general_account_name,
    target.name                 = source.name,
    target.category             = source.category,
    target.code                 = source.code,
    target.ref                  = source.ref,
    target.amount               = source.amount,
    target.unit_amount          = source.unit_amount,
    target.date                 = source.date,
    target.created_at           = source.created_at,
    target.updated_at           = source.updated_at,
    target.dwh_loaded_at        = current_timestamp(),
    target.dwh_source_db        = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_bank_statement (4)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_bank_statement
(
    bank_statement_id INT,

    company_id INT,
    company_name STRING,

    currency_id INT,
    currency_name STRING,

    journal_id INT,
    journal_name STRING,

    name STRING,
    reference STRING,

    balance_start DOUBLE,
    balance_end DOUBLE,
    balance_end_real DOUBLE,

    is_complete BOOLEAN,

    date TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_bank_statement AS target
USING
(
    SELECT
        CAST(bs.id AS INT) AS bank_statement_id,

        CAST(GET_JSON_OBJECT(bs.company_id, '$[0]') AS INT)  AS company_id,
        GET_JSON_OBJECT(bs.company_id, '$[1]')                AS company_name,

        CAST(GET_JSON_OBJECT(bs.currency_id, '$[0]') AS INT) AS currency_id,
        GET_JSON_OBJECT(bs.currency_id, '$[1]')               AS currency_name,

        CAST(GET_JSON_OBJECT(bs.journal_id, '$[0]') AS INT)  AS journal_id,
        GET_JSON_OBJECT(bs.journal_id, '$[1]')                AS journal_name,

        bs.name,
        bs.reference,

        CAST(bs.balance_start AS DOUBLE)    AS balance_start,
        CAST(bs.balance_end AS DOUBLE)      AS balance_end,
        CAST(bs.balance_end_real AS DOUBLE) AS balance_end_real,

        CAST(bs.is_complete AS BOOLEAN) AS is_complete,

        bs.date,

        bs.create_date AS created_at,
        bs.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        bs.dwh_source_db

    FROM bronze.account_bank_statement bs
    WHERE bs.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_bank_statement'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.bank_statement_id = source.bank_statement_id

WHEN MATCHED THEN UPDATE SET
    target.company_id       = source.company_id,
    target.company_name     = source.company_name,
    target.currency_id      = source.currency_id,
    target.currency_name    = source.currency_name,
    target.journal_id       = source.journal_id,
    target.journal_name     = source.journal_name,
    target.name             = source.name,
    target.reference        = source.reference,
    target.balance_start    = source.balance_start,
    target.balance_end      = source.balance_end,
    target.balance_end_real = source.balance_end_real,
    target.is_complete      = source.is_complete,
    target.date             = source.date,
    target.created_at       = source.created_at,
    target.updated_at       = source.updated_at,
    target.dwh_loaded_at    = current_timestamp(),
    target.dwh_source_db    = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_bank_statement_line (5)
-- =====================================================
CREATE TABLE IF NOT EXISTS silver.account_bank_statement_line
(
    bank_statement_line_id INT,

    statement_id INT,

    move_id INT,
    move_name STRING,

    journal_id INT,
    journal_name STRING,

    company_id INT,
    company_name STRING,

    partner_id INT,
    partner_name STRING,

    currency_id INT,
    currency_name STRING,

    foreign_currency_id INT,
    foreign_currency_name STRING,

    sequence INT,
    account_number STRING,
    transaction_type STRING,
    payment_ref STRING,

    amount DOUBLE,
    amount_currency DOUBLE,
    amount_residual DOUBLE,

    is_reconciled BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_bank_statement_line AS target
USING
(
    SELECT
        CAST(bsl.id AS INT) AS bank_statement_line_id,

        try_cast(bsl.statement_id AS INT)                                    AS statement_id,
        CAST(GET_JSON_OBJECT(bsl.move_id, '$[0]') AS INT)                                         AS move_id,
        GET_JSON_OBJECT(bsl.move_id, '$[1]')                          AS move_name,

        CAST(GET_JSON_OBJECT(bsl.journal_id, '$[0]') AS INT)            AS journal_id,
        GET_JSON_OBJECT(bsl.journal_id, '$[1]')                          AS journal_name,

        CAST(GET_JSON_OBJECT(bsl.company_id, '$[0]') AS INT)            AS company_id,
        GET_JSON_OBJECT(bsl.company_id, '$[1]')                          AS company_name,

        CAST(GET_JSON_OBJECT(bsl.partner_id, '$[0]') AS INT)            AS partner_id,
        GET_JSON_OBJECT(bsl.partner_id, '$[1]')                          AS partner_name,

        CAST(GET_JSON_OBJECT(bsl.currency_id, '$[0]') AS INT)           AS currency_id,
        GET_JSON_OBJECT(bsl.currency_id, '$[1]')                         AS currency_name,

        CAST(GET_JSON_OBJECT(bsl.foreign_currency_id, '$[0]') AS INT)   AS foreign_currency_id,
        GET_JSON_OBJECT(bsl.foreign_currency_id, '$[1]')                 AS foreign_currency_name,

        CAST(bsl.sequence AS INT) AS sequence,
        bsl.account_number,
        bsl.transaction_type,
        bsl.payment_ref,

        CAST(bsl.amount AS DOUBLE)          AS amount,
        CAST(bsl.amount_currency AS DOUBLE) AS amount_currency,
        CAST(bsl.amount_residual AS DOUBLE) AS amount_residual,

        CAST(bsl.is_reconciled AS BOOLEAN) AS is_reconciled,

        bsl.create_date AS created_at,
        bsl.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        bsl.dwh_source_db

    FROM bronze.account_bank_statement_line bsl
    WHERE bsl.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_bank_statement_line'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.bank_statement_line_id = source.bank_statement_line_id

WHEN MATCHED THEN UPDATE SET
    target.statement_id          = source.statement_id,
    target.move_id               = source.move_id,
    target.move_name             = source.move_name,
    target.journal_id            = source.journal_id,
    target.journal_name          = source.journal_name,
    target.company_id            = source.company_id,
    target.company_name          = source.company_name,
    target.partner_id            = source.partner_id,
    target.partner_name          = source.partner_name,
    target.currency_id           = source.currency_id,
    target.currency_name         = source.currency_name,
    target.foreign_currency_id   = source.foreign_currency_id,
    target.foreign_currency_name = source.foreign_currency_name,
    target.sequence              = source.sequence,
    target.account_number        = source.account_number,
    target.transaction_type      = source.transaction_type,
    target.payment_ref           = source.payment_ref,
    target.amount                = source.amount,
    target.amount_currency       = source.amount_currency,
    target.amount_residual       = source.amount_residual,
    target.is_reconciled         = source.is_reconciled,
    target.created_at            = source.created_at,
    target.updated_at            = source.updated_at,
    target.dwh_loaded_at         = current_timestamp(),
    target.dwh_source_db         = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_move (6)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_move
(
    account_move_id INT,

    journal_id INT,
    journal_name STRING,

    company_id INT,
    company_name STRING,

    partner_id INT,
    partner_name STRING,

    partner_shipping_id INT,
    partner_shipping_name STRING,

    currency_id INT,
    currency_name STRING,

    invoice_payment_term_id INT,
    invoice_payment_term_name STRING,

    fiscal_position_id INT,
    fiscal_position_name STRING,

    invoice_user_id INT,
    invoice_user_name STRING,

    reversed_entry_id INT,
    campaign_id INT,
    campaign_name STRING,

    team_id INT,
    team_name STRING,

    sequence_prefix STRING,
    name STRING,
    ref STRING,
    state STRING,
    move_type STRING,
    auto_post STRING,
    payment_reference STRING,
    payment_state STRING,
    invoice_origin STRING,
    invoice_partner_display_name STRING,
    narration STRING,

    invoice_currency_rate DOUBLE,
    amount_untaxed DOUBLE,
    amount_tax DOUBLE,
    amount_total DOUBLE,
    amount_residual DOUBLE,
    amount_untaxed_signed DOUBLE,
    amount_tax_signed DOUBLE,
    amount_total_signed DOUBLE,
    amount_residual_signed DOUBLE,

    always_tax_exigible BOOLEAN,
    checked BOOLEAN,
    posted_before BOOLEAN,
    is_move_sent BOOLEAN,

    date TIMESTAMP,
    invoice_date TIMESTAMP,
    invoice_date_due TIMESTAMP,
    delivery_date TIMESTAMP,
    auto_post_until TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_move AS target
USING
(
    SELECT
        CAST(am.id AS INT) AS account_move_id,

        CAST(GET_JSON_OBJECT(am.journal_id, '$[0]') AS INT)               AS journal_id,
        GET_JSON_OBJECT(am.journal_id, '$[1]')                             AS journal_name,

        CAST(GET_JSON_OBJECT(am.company_id, '$[0]') AS INT)               AS company_id,
        GET_JSON_OBJECT(am.company_id, '$[1]')                             AS company_name,

        CAST(GET_JSON_OBJECT(am.partner_id, '$[0]') AS INT)               AS partner_id,
        GET_JSON_OBJECT(am.partner_id, '$[1]')                             AS partner_name,

        CAST(GET_JSON_OBJECT(am.partner_shipping_id, '$[0]') AS INT)      AS partner_shipping_id,
        GET_JSON_OBJECT(am.partner_shipping_id, '$[1]')                    AS partner_shipping_name,

        CAST(GET_JSON_OBJECT(am.currency_id, '$[0]') AS INT)              AS currency_id,
        GET_JSON_OBJECT(am.currency_id, '$[1]')                            AS currency_name,

        CAST(GET_JSON_OBJECT(am.invoice_payment_term_id, '$[0]') AS INT)  AS invoice_payment_term_id,
        GET_JSON_OBJECT(am.invoice_payment_term_id, '$[1]')                AS invoice_payment_term_name,

        CAST(GET_JSON_OBJECT(am.fiscal_position_id, '$[0]') AS INT)       AS fiscal_position_id,
        GET_JSON_OBJECT(am.fiscal_position_id, '$[1]')                     AS fiscal_position_name,

        CAST(GET_JSON_OBJECT(am.invoice_user_id, '$[0]') AS INT)          AS invoice_user_id,
        GET_JSON_OBJECT(am.invoice_user_id, '$[1]')                        AS invoice_user_name,

        CAST(am.reversed_entry_id AS INT)                                  AS reversed_entry_id,

        CAST(GET_JSON_OBJECT(am.campaign_id, '$[0]') AS INT)              AS campaign_id,
        GET_JSON_OBJECT(am.campaign_id, '$[1]')                            AS campaign_name,

        CAST(GET_JSON_OBJECT(am.team_id, '$[0]') AS INT)                  AS team_id,
        GET_JSON_OBJECT(am.team_id, '$[1]')                                AS team_name,

        am.sequence_prefix,
        am.name,
        am.ref,
        am.state,
        am.move_type,
        am.auto_post,
        am.payment_reference,
        am.payment_state,
        am.invoice_origin,
        am.invoice_partner_display_name,
        am.narration,

        CAST(am.invoice_currency_rate AS DOUBLE) AS invoice_currency_rate,
        CAST(am.amount_untaxed AS DOUBLE)        AS amount_untaxed,
        CAST(am.amount_tax AS DOUBLE)            AS amount_tax,
        CAST(am.amount_total AS DOUBLE)          AS amount_total,
        CAST(am.amount_residual AS DOUBLE)       AS amount_residual,
        CAST(am.amount_untaxed_signed AS DOUBLE) AS amount_untaxed_signed,
        CAST(am.amount_tax_signed AS DOUBLE)     AS amount_tax_signed,
        CAST(am.amount_total_signed AS DOUBLE)   AS amount_total_signed,
        CAST(am.amount_residual_signed AS DOUBLE) AS amount_residual_signed,

        CAST(am.always_tax_exigible AS BOOLEAN) AS always_tax_exigible,
        CAST(am.checked AS BOOLEAN)             AS checked,
        CAST(am.posted_before AS BOOLEAN)       AS posted_before,
        CAST(am.is_move_sent AS BOOLEAN)        AS is_move_sent,

        am.date,
        am.invoice_date,
        am.invoice_date_due,
        am.delivery_date,
        am.auto_post_until,

        am.create_date AS created_at,
        am.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        am.dwh_source_db

    FROM bronze.account_move am
    WHERE am.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_move'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_move_id = source.account_move_id

WHEN MATCHED THEN UPDATE SET
    target.journal_id                  = source.journal_id,
    target.journal_name                = source.journal_name,
    target.company_id                  = source.company_id,
    target.company_name                = source.company_name,
    target.partner_id                  = source.partner_id,
    target.partner_name                = source.partner_name,
    target.partner_shipping_id         = source.partner_shipping_id,
    target.partner_shipping_name       = source.partner_shipping_name,
    target.currency_id                 = source.currency_id,
    target.currency_name               = source.currency_name,
    target.invoice_payment_term_id     = source.invoice_payment_term_id,
    target.invoice_payment_term_name   = source.invoice_payment_term_name,
    target.fiscal_position_id          = source.fiscal_position_id,
    target.fiscal_position_name        = source.fiscal_position_name,
    target.invoice_user_id             = source.invoice_user_id,
    target.invoice_user_name           = source.invoice_user_name,
    target.reversed_entry_id           = source.reversed_entry_id,
    target.campaign_id                 = source.campaign_id,
    target.campaign_name               = source.campaign_name,
    target.team_id                     = source.team_id,
    target.team_name                   = source.team_name,
    target.sequence_prefix             = source.sequence_prefix,
    target.name                        = source.name,
    target.ref                         = source.ref,
    target.state                       = source.state,
    target.move_type                   = source.move_type,
    target.auto_post                   = source.auto_post,
    target.payment_reference           = source.payment_reference,
    target.payment_state               = source.payment_state,
    target.invoice_origin              = source.invoice_origin,
    target.invoice_partner_display_name = source.invoice_partner_display_name,
    target.narration                   = source.narration,
    target.invoice_currency_rate       = source.invoice_currency_rate,
    target.amount_untaxed              = source.amount_untaxed,
    target.amount_tax                  = source.amount_tax,
    target.amount_total                = source.amount_total,
    target.amount_residual             = source.amount_residual,
    target.amount_untaxed_signed       = source.amount_untaxed_signed,
    target.amount_tax_signed           = source.amount_tax_signed,
    target.amount_total_signed         = source.amount_total_signed,
    target.amount_residual_signed      = source.amount_residual_signed,
    target.always_tax_exigible         = source.always_tax_exigible,
    target.checked                     = source.checked,
    target.posted_before               = source.posted_before,
    target.is_move_sent                = source.is_move_sent,
    target.date                        = source.date,
    target.invoice_date                = source.invoice_date,
    target.invoice_date_due            = source.invoice_date_due,
    target.delivery_date               = source.delivery_date,
    target.auto_post_until             = source.auto_post_until,
    target.created_at                  = source.created_at,
    target.updated_at                  = source.updated_at,
    target.dwh_loaded_at               = current_timestamp(),
    target.dwh_source_db               = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_move_line (7)
-- =====================================================
CREATE TABLE IF NOT EXISTS silver.account_move_line
(
    account_move_line_id INT,

    move_id INT,
    account_id INT,
    account_name STRING,

    journal_id INT,
    journal_name STRING,

    company_id INT,
    company_name STRING,

    currency_id INT,
    currency_name STRING,

    partner_id INT,
    partner_name STRING,

    product_id INT,
    product_name STRING,

    product_uom_id INT,
    product_uom_name STRING,

    tax_line_id INT,
    tax_line_name STRING,

    tax_group_id INT,
    tax_group_name STRING,

    payment_id INT,
    payment_name STRING,
    sequence INT,
    move_name STRING,
    parent_state STRING,
    ref STRING,
    name STRING,
    display_type STRING,
    analytic_distribution STRING,

    debit DOUBLE,
    credit DOUBLE,
    balance DOUBLE,
    amount_currency DOUBLE,
    tax_base_amount DOUBLE,
    amount_residual DOUBLE,
    quantity DOUBLE,
    price_unit DOUBLE,
    price_subtotal DOUBLE,
    price_total DOUBLE,
    discount DOUBLE,

    is_storno BOOLEAN,
    reconciled BOOLEAN,
    is_downpayment BOOLEAN,

    date TIMESTAMP,
    invoice_date TIMESTAMP,
    date_maturity TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_move_line AS target
USING
(
    SELECT
        CAST(aml.id AS INT)      AS account_move_line_id,

        try_cast(GET_JSON_OBJECT(aml.move_id, '$[0]') AS INT)     AS move_id,

        CAST(GET_JSON_OBJECT(aml.account_id, '$[0]') AS INT)       AS account_id,
        GET_JSON_OBJECT(aml.account_id, '$[1]')                     AS account_name,

        CAST(GET_JSON_OBJECT(aml.journal_id, '$[0]') AS INT)       AS journal_id,
        GET_JSON_OBJECT(aml.journal_id, '$[1]')                     AS journal_name,

        CAST(GET_JSON_OBJECT(aml.company_id, '$[0]') AS INT)       AS company_id,
        GET_JSON_OBJECT(aml.company_id, '$[1]')                     AS company_name,

        CAST(GET_JSON_OBJECT(aml.currency_id, '$[0]') AS INT)      AS currency_id,
        GET_JSON_OBJECT(aml.currency_id, '$[1]')                    AS currency_name,

        CAST(GET_JSON_OBJECT(aml.partner_id, '$[0]') AS INT)       AS partner_id,
        GET_JSON_OBJECT(aml.partner_id, '$[1]')                     AS partner_name,

        CAST(GET_JSON_OBJECT(aml.product_id, '$[0]') AS INT)       AS product_id,
        GET_JSON_OBJECT(aml.product_id, '$[1]')                     AS product_name,

        CAST(GET_JSON_OBJECT(aml.product_uom_id, '$[0]') AS INT)   AS product_uom_id,
        GET_JSON_OBJECT(aml.product_uom_id, '$[1]')                 AS product_uom_name,

        CAST(GET_JSON_OBJECT(aml.tax_line_id, '$[0]') AS INT)      AS tax_line_id,
        GET_JSON_OBJECT(aml.tax_line_id, '$[1]')                    AS tax_line_name,

        CAST(GET_JSON_OBJECT(aml.tax_group_id, '$[0]') AS INT)     AS tax_group_id,
        GET_JSON_OBJECT(aml.tax_group_id, '$[1]')                   AS tax_group_name,

        try_cast(GET_JSON_OBJECT(aml.payment_id, '$[0]') AS INT)   AS payment_id,
        GET_JSON_OBJECT(aml.payment_id, '$[1]')                     AS payment_name,

        CAST(aml.sequence AS INT) AS sequence,
        aml.move_name,
        aml.parent_state,
        aml.ref,
        aml.name,
        aml.display_type,
        aml.analytic_distribution,

        CAST(aml.debit AS DOUBLE)               AS debit,
        CAST(aml.credit AS DOUBLE)              AS credit,
        CAST(aml.balance AS DOUBLE)             AS balance,
        CAST(aml.amount_currency AS DOUBLE)     AS amount_currency,
        CAST(aml.tax_base_amount AS DOUBLE)     AS tax_base_amount,
        CAST(aml.amount_residual AS DOUBLE)     AS amount_residual,
        CAST(aml.quantity AS DOUBLE)            AS quantity,
        CAST(aml.price_unit AS DOUBLE)          AS price_unit,
        CAST(aml.price_subtotal AS DOUBLE)      AS price_subtotal,
        CAST(aml.price_total AS DOUBLE)         AS price_total,
        CAST(aml.discount AS DOUBLE)            AS discount,

        CAST(aml.is_storno AS BOOLEAN)      AS is_storno,
        CAST(aml.reconciled AS BOOLEAN)     AS reconciled,
        CAST(aml.is_downpayment AS BOOLEAN) AS is_downpayment,

        aml.date,
        aml.invoice_date,
        aml.date_maturity,

        aml.create_date AS created_at,
        aml.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        aml.dwh_source_db

    FROM bronze.account_move_line aml
    WHERE aml.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_move_line'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_move_line_id = source.account_move_line_id

WHEN MATCHED THEN UPDATE SET
    target.move_id               = source.move_id,
    target.account_id            = source.account_id,
    target.account_name          = source.account_name,
    target.journal_id            = source.journal_id,
    target.journal_name          = source.journal_name,
    target.company_id            = source.company_id,
    target.company_name          = source.company_name,
    target.currency_id           = source.currency_id,
    target.currency_name         = source.currency_name,
    target.partner_id            = source.partner_id,
    target.partner_name          = source.partner_name,
    target.product_id            = source.product_id,
    target.product_name          = source.product_name,
    target.product_uom_id        = source.product_uom_id,
    target.product_uom_name      = source.product_uom_name,
    target.tax_line_id           = source.tax_line_id,
    target.tax_line_name         = source.tax_line_name,
    target.tax_group_id          = source.tax_group_id,
    target.tax_group_name        = source.tax_group_name,
    target.payment_id            = source.payment_id,
    target.sequence              = source.sequence,
    target.move_name             = source.move_name,
    target.parent_state          = source.parent_state,
    target.ref                   = source.ref,
    target.name                  = source.name,
    target.display_type          = source.display_type,
    target.analytic_distribution = source.analytic_distribution,
    target.debit                 = source.debit,
    target.credit                = source.credit,
    target.balance               = source.balance,
    target.amount_currency       = source.amount_currency,
    target.tax_base_amount       = source.tax_base_amount,
    target.amount_residual       = source.amount_residual,
    target.quantity              = source.quantity,
    target.price_unit            = source.price_unit,
    target.price_subtotal        = source.price_subtotal,
    target.price_total           = source.price_total,
    target.discount              = source.discount,
    target.is_storno             = source.is_storno,
    target.reconciled            = source.reconciled,
    target.is_downpayment        = source.is_downpayment,
    target.date                  = source.date,
    target.invoice_date          = source.invoice_date,
    target.date_maturity         = source.date_maturity,
    target.created_at            = source.created_at,
    target.updated_at            = source.updated_at,
    target.dwh_loaded_at         = current_timestamp(),
    target.dwh_source_db         = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;

In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_journal (8)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_journal
(
    account_journal_id INT,

    company_id INT,
    company_name STRING,

    currency_id INT,
    currency_name STRING,

    default_account_id INT,
    default_account_name STRING,

    sequence INT,
    code STRING,
    type STRING,
    name STRING,
    bank_statements_source STRING,
    invoice_reference_type STRING,
    invoice_reference_model STRING,

    active BOOLEAN,
    is_self_billing BOOLEAN,
    restrict_mode_hash_table BOOLEAN,
    refund_sequence BOOLEAN,
    payment_sequence BOOLEAN,
    show_on_dashboard BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_journal AS target
USING
(
    SELECT
        CAST(aj.id AS INT) AS account_journal_id,

        CAST(GET_JSON_OBJECT(aj.company_id, '$[0]') AS INT)          AS company_id,
        GET_JSON_OBJECT(aj.company_id, '$[1]')                        AS company_name,

        CAST(GET_JSON_OBJECT(aj.currency_id, '$[0]') AS INT)         AS currency_id,
        GET_JSON_OBJECT(aj.currency_id, '$[1]')                       AS currency_name,

        CAST(GET_JSON_OBJECT(aj.default_account_id, '$[0]') AS INT)  AS default_account_id,
        GET_JSON_OBJECT(aj.default_account_id, '$[1]')                AS default_account_name,

        CAST(aj.sequence AS INT) AS sequence,
        aj.code,
        aj.type,
        aj.name,
        aj.bank_statements_source,
        aj.invoice_reference_type,
        aj.invoice_reference_model,

        CAST(aj.active AS BOOLEAN)                   AS active,
        CAST(aj.is_self_billing AS BOOLEAN)          AS is_self_billing,
        CAST(aj.restrict_mode_hash_table AS BOOLEAN) AS restrict_mode_hash_table,
        CAST(aj.refund_sequence AS BOOLEAN)          AS refund_sequence,
        CAST(aj.payment_sequence AS BOOLEAN)         AS payment_sequence,
        CAST(aj.show_on_dashboard AS BOOLEAN)        AS show_on_dashboard,

        aj.create_date AS created_at,
        aj.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        aj.dwh_source_db

    FROM bronze.account_journal aj
    WHERE aj.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_journal'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_journal_id = source.account_journal_id

WHEN MATCHED THEN UPDATE SET
    target.company_id              = source.company_id,
    target.company_name            = source.company_name,
    target.currency_id             = source.currency_id,
    target.currency_name           = source.currency_name,
    target.default_account_id      = source.default_account_id,
    target.default_account_name    = source.default_account_name,
    target.sequence                = source.sequence,
    target.code                    = source.code,
    target.type                    = source.type,
    target.name                    = source.name,
    target.bank_statements_source  = source.bank_statements_source,
    target.invoice_reference_type  = source.invoice_reference_type,
    target.invoice_reference_model = source.invoice_reference_model,
    target.active                  = source.active,
    target.is_self_billing         = source.is_self_billing,
    target.restrict_mode_hash_table = source.restrict_mode_hash_table,
    target.refund_sequence         = source.refund_sequence,
    target.payment_sequence        = source.payment_sequence,
    target.show_on_dashboard       = source.show_on_dashboard,
    target.created_at              = source.created_at,
    target.updated_at              = source.updated_at,
    target.dwh_loaded_at           = current_timestamp(),
    target.dwh_source_db           = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;



In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_tax (9)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_tax
(
    account_tax_id INT,

    company_id INT,
    company_name STRING,

    tax_group_id INT,
    tax_group_name STRING,

    country_id INT,
    country_name STRING,

    sequence INT,
    type_tax_use STRING,
    tax_scope STRING,
    amount_type STRING,
    tax_exigibility STRING,
    name STRING,
    description STRING,
    invoice_label STRING,
    ubl_cii_tax_category_code STRING,
    l10n_eg_eta_code STRING,

    amount DOUBLE,

    is_domestic BOOLEAN,
    active BOOLEAN,
    include_base_amount BOOLEAN,
    is_base_affected BOOLEAN,
    analytic BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_tax AS target
USING
(
    SELECT
        CAST(at.id AS INT) AS account_tax_id,

        CAST(GET_JSON_OBJECT(at.company_id, '$[0]') AS INT)    AS company_id,
        GET_JSON_OBJECT(at.company_id, '$[1]')                  AS company_name,

        CAST(GET_JSON_OBJECT(at.tax_group_id, '$[0]') AS INT)  AS tax_group_id,
        GET_JSON_OBJECT(at.tax_group_id, '$[1]')                AS tax_group_name,

        CAST(GET_JSON_OBJECT(at.country_id, '$[0]') AS INT)    AS country_id,
        GET_JSON_OBJECT(at.country_id, '$[1]')                  AS country_name,

        CAST(at.sequence AS INT) AS sequence,
        at.type_tax_use,
        at.tax_scope,
        at.amount_type,
        at.tax_exigibility,
        at.name,
        at.description,
        at.invoice_label,
        at.ubl_cii_tax_category_code,
        at.l10n_eg_eta_code,

        CAST(at.amount AS DOUBLE) AS amount,

        CAST(at.is_domestic AS BOOLEAN)         AS is_domestic,
        CAST(at.active AS BOOLEAN)              AS active,
        CAST(at.include_base_amount AS BOOLEAN) AS include_base_amount,
        CAST(at.is_base_affected AS BOOLEAN)    AS is_base_affected,
        CAST(at.analytic AS BOOLEAN)            AS analytic,

        at.create_date AS created_at,
        at.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        at.dwh_source_db

    FROM bronze.account_tax at
    WHERE at.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_tax'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_tax_id = source.account_tax_id

WHEN MATCHED THEN UPDATE SET
    target.company_id              = source.company_id,
    target.company_name            = source.company_name,
    target.tax_group_id            = source.tax_group_id,
    target.tax_group_name          = source.tax_group_name,
    target.country_id              = source.country_id,
    target.country_name            = source.country_name,
    target.sequence                = source.sequence,
    target.type_tax_use            = source.type_tax_use,
    target.tax_scope               = source.tax_scope,
    target.amount_type             = source.amount_type,
    target.tax_exigibility         = source.tax_exigibility,
    target.name                    = source.name,
    target.description             = source.description,
    target.invoice_label           = source.invoice_label,
    target.ubl_cii_tax_category_code = source.ubl_cii_tax_category_code,
    target.l10n_eg_eta_code        = source.l10n_eg_eta_code,
    target.amount                  = source.amount,
    target.is_domestic             = source.is_domestic,
    target.active                  = source.active,
    target.include_base_amount     = source.include_base_amount,
    target.is_base_affected        = source.is_base_affected,
    target.analytic                = source.analytic,
    target.created_at              = source.created_at,
    target.updated_at              = source.updated_at,
    target.dwh_loaded_at           = current_timestamp(),
    target.dwh_source_db           = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;



In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_payment (10)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_payment
(
    account_payment_id INT,

    move_id INT,
    move_name STRING,

    journal_id INT,
    journal_name STRING,

    company_id INT,
    company_name STRING,

    partner_id INT,
    partner_name STRING,

    currency_id INT,
    currency_name STRING,

    source_payment_id INT,
    payment_method_id INT,
    payment_method_name STRING,

    name STRING,
    state STRING,
    payment_type STRING,
    partner_type STRING,
    memo STRING,
    payment_reference STRING,

    amount DOUBLE,
    amount_company_currency_signed DOUBLE,

    is_reconciled BOOLEAN,
    is_matched BOOLEAN,
    is_sent BOOLEAN,

    date TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_payment AS target
USING
(
    SELECT
        CAST(ap.id AS INT)      AS account_payment_id,

        CAST(GET_JSON_OBJECT(ap.move_id, '$[0]') AS INT)           AS move_id,
        GET_JSON_OBJECT(ap.move_id, '$[1]')          AS move_name,

        CAST(GET_JSON_OBJECT(ap.journal_id, '$[0]') AS INT)          AS journal_id,
        GET_JSON_OBJECT(ap.journal_id, '$[1]')                        AS journal_name,

        CAST(GET_JSON_OBJECT(ap.company_id, '$[0]') AS INT)          AS company_id,
        GET_JSON_OBJECT(ap.company_id, '$[1]')                        AS company_name,

        CAST(GET_JSON_OBJECT(ap.partner_id, '$[0]') AS INT)          AS partner_id,
        GET_JSON_OBJECT(ap.partner_id, '$[1]')                        AS partner_name,

        CAST(GET_JSON_OBJECT(ap.currency_id, '$[0]') AS INT)         AS currency_id,
        GET_JSON_OBJECT(ap.currency_id, '$[1]')                       AS currency_name,

        CAST(ap.source_payment_id AS INT)                             AS source_payment_id,

        CAST(GET_JSON_OBJECT(ap.payment_method_id, '$[0]') AS INT)   AS payment_method_id,
        GET_JSON_OBJECT(ap.payment_method_id, '$[1]')                 AS payment_method_name,

        ap.name,
        ap.state,
        ap.payment_type,
        ap.partner_type,
        ap.memo,
        ap.payment_reference,

        CAST(ap.amount AS DOUBLE)                          AS amount,
        CAST(ap.amount_company_currency_signed AS DOUBLE)  AS amount_company_currency_signed,

        CAST(ap.is_reconciled AS BOOLEAN) AS is_reconciled,
        CAST(ap.is_matched AS BOOLEAN)    AS is_matched,
        CAST(ap.is_sent AS BOOLEAN)       AS is_sent,

        ap.date,

        ap.create_date AS created_at,
        ap.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        ap.dwh_source_db

    FROM bronze.account_payment ap
    WHERE ap.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_payment'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_payment_id = source.account_payment_id

WHEN MATCHED THEN UPDATE SET
    target.move_id                         = source.move_id,
    target.move_name                         = source.move_name,
    target.journal_id                      = source.journal_id,
    target.journal_name                    = source.journal_name,
    target.company_id                      = source.company_id,
    target.company_name                    = source.company_name,
    target.partner_id                      = source.partner_id,
    target.partner_name                    = source.partner_name,
    target.currency_id                     = source.currency_id,
    target.currency_name                   = source.currency_name,
    target.source_payment_id               = source.source_payment_id,
    target.payment_method_id               = source.payment_method_id,
    target.payment_method_name             = source.payment_method_name,
    target.name                            = source.name,
    target.state                           = source.state,
    target.payment_type                    = source.payment_type,
    target.partner_type                    = source.partner_type,
    target.memo                            = source.memo,
    target.payment_reference               = source.payment_reference,
    target.amount                          = source.amount,
    target.amount_company_currency_signed  = source.amount_company_currency_signed,
    target.is_reconciled                   = source.is_reconciled,
    target.is_matched                      = source.is_matched,
    target.is_sent                         = source.is_sent,
    target.date                            = source.date,
    target.created_at                      = source.created_at,
    target.updated_at                      = source.updated_at,
    target.dwh_loaded_at                   = current_timestamp(),
    target.dwh_source_db                   = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Table: silver.account_payment_term (11)
-- =====================================================

CREATE TABLE IF NOT EXISTS silver.account_payment_term
(
    account_payment_term_id INT,

    company_id INT,
    company_name STRING,

    sequence INT,
    discount_days INT,

    early_pay_discount_computation STRING,
    name STRING,
    note STRING,

    discount_percentage DOUBLE,

    active BOOLEAN,
    display_on_invoice BOOLEAN,
    early_discount BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- MERGE

MERGE INTO silver.account_payment_term AS target
USING
(
    SELECT
        CAST(apt.id AS INT) AS account_payment_term_id,

        CAST(GET_JSON_OBJECT(apt.company_id, '$[0]') AS INT) AS company_id,
        GET_JSON_OBJECT(apt.company_id, '$[1]')               AS company_name,

        CAST(apt.sequence AS INT)      AS sequence,
        CAST(apt.discount_days AS INT) AS discount_days,

        apt.early_pay_discount_computation,
        apt.name,
        apt.note,

        CAST(apt.discount_percentage AS DOUBLE) AS discount_percentage,

        CAST(apt.active AS BOOLEAN)              AS active,
        CAST(apt.display_on_invoice AS BOOLEAN)  AS display_on_invoice,
        CAST(apt.early_discount AS BOOLEAN)      AS early_discount,

        apt.create_date AS created_at,
        apt.write_date  AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        apt.dwh_source_db

    FROM bronze.account_payment_term apt
    WHERE apt.write_date > COALESCE
    (
        (SELECT last_write_date FROM silver.load_tracking WHERE table_name = 'account_payment_term'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.account_payment_term_id = source.account_payment_term_id

WHEN MATCHED THEN UPDATE SET
    target.company_id                      = source.company_id,
    target.company_name                    = source.company_name,
    target.sequence                        = source.sequence,
    target.discount_days                   = source.discount_days,
    target.early_pay_discount_computation  = source.early_pay_discount_computation,
    target.name                            = source.name,
    target.note                            = source.note,
    target.discount_percentage             = source.discount_percentage,
    target.active                          = source.active,
    target.display_on_invoice              = source.display_on_invoice,
    target.early_discount                  = source.early_discount,
    target.created_at                      = source.created_at,
    target.updated_at                      = source.updated_at,
    target.dwh_loaded_at                   = current_timestamp(),
    target.dwh_source_db                   = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;